In [1]:
from importlib import reload
import torch
import numpy as np
import time
torch.set_default_dtype(torch.float64)
np.random.seed(2)
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
print(device)
import numpy as np
import sys # Add the module path. Windows and Linux use different path separators; the raw string prevents escape sequences such as \n from being interpreted.
sys.path.insert(0, r"../main_code/3d")
for _mod in ['generate_data', 'mesh', 'adaptive_int', 'visual', 'matrix_assemble', 'inverse_solver', 'mea_3d', 'net_3d', 'source_eval', 'error_general_3d', 'main', 'bound_detect', 'q_filter']:
    sys.modules.pop(_mod, None)
import generate_data,mesh,adaptive_int,visual,matrix_assemble,inverse_solver,mea_3d,net_3d,source_eval,error_general_3d,main
CUDA_LAUNCH_BLOCKING=3

def ana_S(x, center,R, r, S0):
    x1=x[:,0]
    x2=x[:,1]
    x3=x[:,2]
    cx, cy, cz = center[0,0],center[0,1],center[0,2]
    
    # Compute whether each point lies inside the torus relative to the shifted center.
    
    x_rel, y_rel, z_rel = x1 - cx, x2 - cy, x3 - cz
    condition = (np.sqrt(x_rel**2 + y_rel**2) - R)**2 + z_rel**2 <= r**2
    
    # Use np.where to return S0 or 0 according to the condition.
    return np.where(condition, S0, 0.0)

cuda:3


In [2]:
import copy
Nx,Ny,Nz=1,1,1
Ix,Iy,Iz=4,4,4  # Number of grid cells in the x direction
nx,ny,nz=3,3,3  # Gauss points per cell in the x direction
kk=[1,5,9,13,17,21,25,29,33,37,41,45,49,53,57,61,65,69,73,77,81]
R_true = 0.25            # Major radius R
r_true = 0.15            # Minor radius r
S0= 1.0  
tau_x_min,tau_x_max,tau_y_min,tau_y_max,tau_z_min,tau_z_max=-0.5,0.5,-0.5,0.5,-0.5,0.5
b_x_min,b_x_max,b_y_min,b_y_max,b_z_min,b_z_max=-0.75,0.75,-0.75,0.75,-0.75,0.75 # On boundary Gamma
num_batches_gauss=1
b_n=10
points_b=mea_3d.generate_cube_surface(b_x_min,b_x_max,b_y_min,b_y_max,b_z_min,b_z_max,b_n) # Collect data on boundary Gamma
center_true=np.array([[0.0,0.0,0.0]])
number_gene=50
eps=0.05
batch_number_rec_mea,mea=2,1
cupy_device = device.index if device.type == "cuda" else 0
num_batches_gauss=4
num_batches_appr=10
num_batches_mea=1

In [3]:
F_diri_5=np.load("./noise=5%/F_diri_5%.npy")


In [4]:
M=5200
af="Tanh"   # Initial activation setting
R_m=20
models0 = net_3d.local_rep(in_features=3, out_features=1, hidden_layers=1, M=M, x_max=tau_x_min, x_min=tau_x_max, y_max=tau_y_max, y_min=tau_y_min,z_min=tau_z_min,z_max=tau_z_max, r_min=[], r_max=[],R_min=[],R_max=[],b1_min=[],b1_max=[],b2_min=[],b2_max=[],b3_min=[],b3_max=[],K_min=[],K_max=[],R_m_for_init=R_m,af=af,Shape=[],device=device).to(device)

In [5]:
models0=torch.load("./noise=5%/model_tanh_4*10**-4.pth", map_location=device)

In [ ]:
Ix,Iy,Iz=4,4,4  # Number of grid cells in the x direction
nx,ny,nz=3,3,3  # Gauss points per cell in the x direction
ratio1=[0.5,0.5,0.5]
cells= mesh.create_initial_grid(tau_x_min,tau_x_max,tau_y_min,tau_y_max,tau_z_min,tau_z_max,Ix,nx,points_b) # Create the initial grid
initial_cells=copy.deepcopy(cells) # Make a copy
delta=10**(-3)

lamb_regu=2*np.logspace(-2,-1,1)
Qx,Qy,Qz=50,50,50
iter_int=3
max_level=4
current_maxiter=1
refine_threshold_S = 1/100
refine_threshold_grad = 1/300
condition="Dirichlet"
cells_store,refinement_stats_store,w_,point_number,g_store,g_S,S_num_store,S_l2,S_basis_store=main.ada_int(iter_int,delta,cells,nx,models0,M,af,kk,F_diri_5,tau_x_min,tau_x_max,tau_y_min,tau_y_max,tau_z_min,tau_z_max,center_true,R_true,r_true,S0,ana_S,points_b,lamb_regu,Qx,Qy,Qz,ratio1,condition,device,cupy_device,refine_threshold_S,refine_threshold_grad,current_maxiter,max_level)